In [1]:
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.utils.data
import torchvision
from torchvision import transforms
from torchvision.utils import save_image
import torch_directml

In [2]:
# Гиперпараметры
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(0))
elif torch_directml.is_available():
    device = torch_directml.device()
    print(torch_directml.device_name(0))
else:
    device = torch.device("cpu")
    print("cpu")

image_size = 28 * 28
hidden_size = 400
latent_dim = 20
batch_size = 128
epochs = 30

train_dataset = torchvision.datasets.MNIST(root='./source/',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)
test_dataset = torchvision.datasets.MNIST(root='./source/',
                                          train=False,
                                          transform=transforms.ToTensor(),
                                          download=True)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=True)

AMD Radeon RX 6800S 


In [3]:
# Saving
sample_dir = './source/VAE/'
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)

In [4]:
# VAE
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(image_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc_mu = nn.Linear(hidden_size, latent_dim)
        self.fc_logvar = nn.Linear(hidden_size, latent_dim)
        self.fc2 = nn.Linear(latent_dim, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        self.fc3 = nn.Linear(hidden_size, image_size)
    def encode(self, x):
        h = x.view(-1, image_size)
        h = self.fc1(h)
        h = self.bn1(h)
        h = f.leaky_relu(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    def reparameterize(self, mu, logvar):
        std = torch.exp(logvar / 2)
        eps = torch.randn_like(std)
        return std * eps + mu
    def decode(self, z):
        h = self.fc2(z)
        h = self.bn2(h)
        h = f.leaky_relu(h)
        out = self.fc3(h)
        return out
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        out = self.decode(z)
        return out, mu, logvar

In [5]:
# Objects
model = VAE()
model = model.to(device)

optimizer = torch.optim.Adam(params=model.parameters(),
                            lr=1e-3,
                             weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,
                                                                 T_0=20)

In [6]:
# Functions
def loss_function(reconstructed_image, original_image, mu, logvar, beta=1.0):
    bce = f.binary_cross_entropy_with_logits(reconstructed_image,
                                 original_image.view(-1, image_size),
                                 reduction='sum')
    kld = -0.5 * torch.sum(1.0 + logvar - logvar.exp() - mu.pow(2))
    return bce + beta * kld

def train(epoch):
    model.train()
    train_loss = 0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        reconstructed_images, mu, logvar = model(images)
        beta = min(1.0, epoch / 5.0)
        loss = loss_function(reconstructed_images, images, mu, logvar, beta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if i % 100 == 0:
            print(f"Train Epoch: {epoch + 1}/{epochs}")
            print(f"Batch: {i + 1}/{len(train_loader)}")
            print(f"Loss: {loss / len(images):.3f}")
            print()
    print()
    print(f"===> Train Epoch: {epoch + 1}/{epochs}")
    print(f"===> Loss: {train_loss / len(train_loader.dataset):.3f}")
    print()
    print()
    scheduler.step()

def test(epoch):
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for i, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            reconstructed_images, mu, logvar = model(images)
            loss = loss_function(reconstructed_images, images, mu, logvar)
            test_loss += loss.item()
            if i == 0:
                comparison = torch.cat([images[:5], reconstructed_images.view(batch_size, 1, 28, 28)[:5]])
                save_image(comparison.cpu(), sample_dir + 'reconstruction_' + str(epoch + 1) + '.png', nrow=5)
    print(f"Test Epoch: {epoch + 1}/{epochs}")
    print(f"Loss: {test_loss / len(test_loader.dataset):.3f}")

In [7]:
# Train
for epoch in range(epochs):
    train(epoch)
    test(epoch)
    with torch.no_grad():
        sample = torch.randn(64, latent_dim).to(device)
        generated = torch.sigmoid(model.decode(sample).cpu())
        save_image(generated.view(64, 1, 28, 28), sample_dir + 'sample_' + str(epoch + 1) + '.png')

C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torch\nn\functional.py:3244: UserWarning: The operator 'aten::log_sigmoid_forward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch.binary_cross_entropy_with_logits(input, target, weight, pos_weight, reduction_enum)


Train Epoch: 1/30
Batch: 1/469
Loss: 554.893

Train Epoch: 1/30
Batch: 101/469
Loss: 116.108

Train Epoch: 1/30
Batch: 201/469
Loss: 95.805

Train Epoch: 1/30
Batch: 301/469
Loss: 88.532

Train Epoch: 1/30
Batch: 401/469
Loss: 79.969


===> Train Epoch: 1/30
===> Loss: 109.979


Test Epoch: 1/30
Loss: 187.144
Train Epoch: 2/30
Batch: 1/469
Loss: 97.399

Train Epoch: 2/30
Batch: 101/469
Loss: 93.590

Train Epoch: 2/30
Batch: 201/469
Loss: 89.236

Train Epoch: 2/30
Batch: 301/469
Loss: 89.600

Train Epoch: 2/30
Batch: 401/469
Loss: 92.261


===> Train Epoch: 2/30
===> Loss: 91.221


Test Epoch: 2/30
Loss: 120.907
Train Epoch: 3/30
Batch: 1/469
Loss: 95.975

Train Epoch: 3/30
Batch: 101/469
Loss: 97.363

Train Epoch: 3/30
Batch: 201/469
Loss: 96.301

Train Epoch: 3/30
Batch: 301/469
Loss: 95.408

Train Epoch: 3/30
Batch: 401/469
Loss: 90.839


===> Train Epoch: 3/30
===> Loss: 95.152


Test Epoch: 3/30
Loss: 113.304
Train Epoch: 4/30
Batch: 1/469
Loss: 101.378

Train Epoch: 4/30
Batch: 10